# Thème Numéro 1 - La Perception de Soi

## Breakdown:
- Q1 Les personnes qui s'évaluent mieux obtiennent-elles plus de matchs?
- Q2 Vaut-il mieux être confiant ou réaliste?
- Q3 Le succès lors du speed-dating (nombre de matchs obtenus) influence-t-il la perception de soi après l'événement?

## Question 3 - Le succès influence-t-il la perception de soi ?
Le succès lors du speed dating (nombre de matchs obtenus) influence-t-il
la perception de soi après l'événement ?

- **H0** : il n'y a pas de relation entre le nombre de matchs obtenus et
  la perception de soi après l'événement
- **H1** : il existe une relation positive entre le nombre de matchs obtenus
  et la perception de soi après l'événement (plus de matchs → perception de soi plus élevée)

On analysera cette relation à deux moments :
- **T2** : juste après l'événement
- **T3** : 3-4 semaines après l'événement

Seuil de significativité : α = 0.05

## 0. Chargement des données

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import plotly.express as px

In [ ]:
df = pd.read_csv("Speed+Dating+Data.csv", encoding="MacRoman")
# display(df.head())
df.info()
# df.describe()

## 1. Création des variables
On calcule pour chaque individu :
- **Nombre de matchs** : somme des rencontres où `match == 1`
- **Self-perception T2** : moyenne globale des auto-évaluations juste après l'événement
  (attr3_2, sinc3_2, intel3_2, fun3_2, amb3_2)
- **Self-perception T3** : moyenne globale des auto-évaluations 3-4 semaines après
  (attr3_3, sinc3_3, intel3_3, fun3_3, amb3_3)

In [ ]:
cols_self_t2 = ['attr3_2', 'sinc3_2', 'intel3_2', 'fun3_2', 'amb3_2']
cols_self_t3 = ['attr3_3', 'sinc3_3', 'intel3_3', 'fun3_3', 'amb3_3']

df_1_3 = df.copy()

# Imputation des valeurs nulles par la médiane de chaque individu
for col in cols_self_t2 + cols_self_t3:
    df_1_3[col] = df_1_3.groupby('iid')[col].transform(
        lambda x: x.fillna(x.median())
    )

# Création de per_person
per_person = df_1_3.groupby('iid').agg(
    nb_matchs=('match', 'sum')
).reset_index()

# Self-perception T2 et T3
per_person['self_perception_t2'] = df_1_3.groupby('iid')[cols_self_t2].mean().mean(axis=1).values
per_person['self_perception_t3'] = df_1_3.groupby('iid')[cols_self_t3].mean().mean(axis=1).values

# Suppression des NaN résiduels
per_person = per_person.dropna(subset=['self_perception_t2', 'self_perception_t3'])

per_person.head()

## 2. Corrélation de Pearson
On utilise la corrélation de Pearson car les deux variables sont continues.
Elle mesure la force et la direction de la relation linéaire entre le nombre
de matchs et la perception de soi après l'événement.

### T2 : juste après l'événement

In [ ]:
clean_t2 = per_person[['nb_matchs', 'self_perception_t2']].dropna()

r_t2, p_t2 = stats.pearsonr(clean_t2['nb_matchs'], clean_t2['self_perception_t2'])

print(f"Corrélation de Pearson : r = {r_t2:.3f}")
print(f"p-value : {p_t2:.4f}")

if p_t2 < 0.05:
    print("\n→ H0 rejetée : relation significative entre matchs et perception de soi (T2)")
    if r_t2 > 0:
        print("→ Corrélation positive : plus de matchs = meilleure perception de soi")
    else:
        print("→ Corrélation négative (inattendu)")
else:
    print("\n→ H0 non rejetée : pas de relation significative (T2)")

### T3 : 3-4 semaines après l'événement

In [ ]:
clean_t3 = per_person[['nb_matchs', 'self_perception_t3']].dropna()

r_t3, p_t3 = stats.pearsonr(clean_t3['nb_matchs'], clean_t3['self_perception_t3'])

print(f"Corrélation de Pearson : r = {r_t3:.3f}")
print(f"p-value : {p_t3:.4f}")

if p_t3 < 0.05:
    print("\n→ H0 rejetée : relation significative entre matchs et perception de soi (T3)")
    if r_t3 > 0:
        print("→ Corrélation positive : plus de matchs = meilleure perception de soi")
    else:
        print("→ Corrélation négative (inattendu)")
else:
    print("\n→ H0 non rejetée : pas de relation significative (T3)")

## 3. Visualisation
Les scatter plots ci-dessous illustrent la relation entre le nombre de matchs
et la perception de soi à chaque temps de mesure.
La droite de régression illustre la tendance générale.

In [ ]:
fig_t2 = px.scatter(
    clean_t2,
    x='nb_matchs',
    y='self_perception_t2',
    trendline='ols',
    title="Relation entre nombre de matchs et la perception de soi (T2 - juste après l'événement)",
    labels={
        'nb_matchs': 'Nombre de Matchs',
        'self_perception_t2': "Self-perception juste après l'événement"
    }
)
fig_t2.show()

fig_t3 = px.scatter(
    clean_t3,
    x='nb_matchs',
    y='self_perception_t3',
    trendline='ols',
    title="Relation entre nombre de matchs et la perception de soi (T3 - 3-4 semaines après l'événement)",
    labels={
        'nb_matchs': 'Nombre de Matchs',
        'self_perception_t3': "Self-perception 3-4 semaines après l'événement"
    }
)
fig_t3.show()

## 4. Récapitulatif T2 vs T3

In [ ]:
print("Récapitulatif des corrélations :")
print(f"  Juste après l'événement (T2)  : r = {r_t2:.3f}, p = {p_t2:.4f}")
print(f"  3-4 semaines après (T3)       : r = {r_t3:.3f}, p = {p_t3:.4f}")

## Conclusion

Les deux corrélations sont quasi nulles et non significatives :
- **T2** (juste après) : r = 0.029, p = 0.649
- **T3** (3-4 semaines après) : r = 0.077, p = 0.227

On ne peut rejeter H0 à aucun des deux temps de mesure.

**Le nombre de matchs obtenus n'influence pas la perception de soi, ni immédiatement
après l'événement, ni 3-4 semaines plus tard.**

Comme le confirment visuellement les deux scatter plots, les points sont dispersés
sans tendance claire aux deux temps de mesure — la droite de régression reste
pratiquement horizontale en T2, et très légèrement positive en T3, mais insuffisamment
pour être significative.

### Ce que ça nous dit
La perception de soi semble être une variable **stable et résistante** au succès
romantique à court terme. Même un effet différé (T3) n'apparaît pas, ce qui suggère
qu'une seule expérience de speed dating, aussi réussie soit-elle, ne suffit pas à
modifier durablement l'image que l'on a de soi.

### Limites à considérer
- Un suivi sur une période plus longue pourrait révéler un effet que ces deux temps
  de mesure ne capturent pas
- La perception de soi mesurée ici est une moyenne de plusieurs dimensions
  (attractivité, sincérité, intelligence...) — un effet pourrait exister sur
  une dimension spécifique mais être dilué dans la moyenne globale